# DenseNet169 com validação cruzada

Este caderno registra o experimento da DenseNet169 com validação cruzada estratificada em cinco folds. O objetivo é medir a variação do desempenho da arquitetura antes de compará-la com os outros modelos.

A divisão dos dados segue o mesmo protocolo em todas as rodadas:

- os folds são formados apenas com as imagens reais de `train_real_split.csv`;
- `test_real_split.csv` não participa desta etapa e será usado somente na avaliação final;
- o oversampling é feito depois da separação, apenas sobre o treino do fold;
- o subconjunto de validação conserva a distribuição original;
- não são utilizados DeepSMOTE nem pesos de classe neste experimento.

As probabilidades obtidas nas cinco validações são reunidas em um único arquivo de predições out-of-fold (OOF).

## 1. Preparação do ambiente

In [1]:
import sys
from pathlib import Path

module_dir = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    direct = candidate / "cross_validation_utils.py"
    nested = candidate / "03_transfer_learning" / "cross_validation" / "cross_validation_utils.py"
    if direct.is_file():
        module_dir = direct.parent
        break
    if nested.is_file():
        module_dir = nested.parent
        break

if module_dir is None:
    raise RuntimeError("Não foi possível localizar cross_validation_utils.py")

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from cross_validation_utils import (
    CLASSES,
    CrossValidationConfig,
    balance_training_dataframe,
    build_fold_assignments,
    configure_runtime,
    fold_distribution,
    load_real_training_dataframe,
    run_cross_validation,
)

print("Módulo carregado de:", module_dir)
print("GPUs disponíveis:", configure_runtime(seed=42))


Módulo carregado de: /Users/alexandre/Documents/GitHub/skin-cancer-classification/03_transfer_learning/cross_validation
GPUs disponíveis: []


## 2. Parâmetros do experimento

In [2]:
config = CrossValidationConfig(
    architecture="densenet169",
    image_size=(320, 320),
    batch_size=16,
    epochs=60,
    n_splits=5,
    learning_rate=1e-4,
    early_stopping_patience=8,
    reduce_lr_patience=4,
    seed=42,
    balance_strategy="random_oversampling",
    oversampling_target_count=None,
    backbone_weights="imagenet",
    save_fold_models=False,
)

config


CrossValidationConfig(architecture='densenet169', image_size=(320, 320), batch_size=16, epochs=60, n_splits=5, learning_rate=0.0001, early_stopping_patience=8, reduce_lr_patience=4, seed=42, balance_strategy='random_oversampling', oversampling_target_count=None, backbone_weights='imagenet', save_fold_models=False)

## 3. Verificação dos dados e dos folds

In [3]:
train_real_df = load_real_training_dataframe()
fold_assignments = build_fold_assignments(train_real_df, config)

print("Total de imagens reais para validação cruzada:", len(train_real_df))
print("Distribuição geral:")
display(train_real_df["class_name"].value_counts().reindex(CLASSES).to_frame("quantidade"))
print("Distribuição por fold:")
display(fold_distribution(fold_assignments))

example_fold = 1
example_validation = fold_assignments.loc[fold_assignments["fold"] == example_fold]
example_train = fold_assignments.loc[fold_assignments["fold"] != example_fold]
example_oversampled = balance_training_dataframe(example_train, config, fold=example_fold)

print(f"Treino do fold {example_fold} antes do oversampling:")
display(example_train["class_name"].value_counts().reindex(CLASSES).to_frame("quantidade"))
print(f"Treino do fold {example_fold} depois do oversampling:")
display(example_oversampled["class_name"].value_counts().reindex(CLASSES).to_frame("quantidade"))

assert fold_assignments["image"].nunique() == len(fold_assignments)
assert fold_assignments["fold"].between(1, config.n_splits).all()
assert set(example_oversampled["image"]).isdisjoint(example_validation["image"])
assert example_oversampled["label"].value_counts().nunique() == 1
print("Folds sem sobreposição e treino balanceado somente após a separação.")


Total de imagens reais para validação cruzada: 8012
Distribuição geral:


,quantidade
class_name,
MEL,890
NV,5364
BCC,411
AKIEC,262
BKL,879
DF,92
VASC,114


Distribuição por fold:


class_name,MEL,NV,BCC,AKIEC,BKL,DF,VASC
fold,,,,,,,
1,178,1073,82,52,176,19,23
2,178,1073,82,53,176,18,23
3,178,1073,82,53,176,18,22
4,178,1073,83,52,175,18,23
5,178,1072,82,52,176,19,23


Treino do fold 1 antes do oversampling:


,quantidade
class_name,
MEL,712
NV,4291
BCC,329
AKIEC,210
BKL,703
DF,73
VASC,91


Treino do fold 1 depois do oversampling:


,quantidade
class_name,
MEL,4291
NV,4291
BCC,4291
AKIEC,4291
BKL,4291
DF,4291
VASC,4291


Folds sem sobreposição e treino balanceado somente após a separação.


## 4. Treinamento

Nesta etapa são treinadas cinco instâncias da DenseNet169, uma para cada fold de validação. A execução pode levar várias horas; por isso, vale conferir a distribuição das classes antes de iniciá-la.

In [4]:
results = run_cross_validation(config)



Fold 1/5
Epoch 1/60
1878/1878 ━━━━━━━━━━━━━━━━━━━━ 2558s 1s/step - accuracy: 0.4973 - loss: 1.6012 - val_accuracy: 0.5995 - val_loss: 1.2305 - learning_rate: 1.0000e-04
Epoch 2/60
1878/1878 ━━━━━━━━━━━━━━━━━━━━ 3069s 2s/step - accuracy: 0.6410 - loss: 1.0968 - val_accuracy: 0.6251 - val_loss: 1.1151 - learning_rate: 1.0000e-04
Epoch 3/60
1878/1878 ━━━━━━━━━━━━━━━━━━━━ 3017s 2s/step - accuracy: 0.6977 - loss: 0.9125 - val_accuracy: 0.6650 - val_loss: 1.0187 - learning_rate: 1.0000e-04
Epoch 4/60
1878/1878 ━━━━━━━━━━━━━━━━━━━━ 2959s 2s/step - accuracy: 0.7316 - loss: 0.8189 - val_accuracy: 0.6500 - val_loss: 1.0419 - learning_rate: 1.0000e-04
Epoch 5/60
1878/1878 ━━━━━━━━━━━━━━━━━━━━ 2886s 2s/step - accuracy: 0.7584 - loss: 0.7359 - val_accuracy: 0.6532 - val_loss: 1.0361 - learning_rate: 1.0000e-04
Epoch 6/60
1878/1878 ━━━━━━━━━━━━━━━━━━━━ 2486s 1s/step - accuracy: 0.7755 - loss: 0.6850 - val_accuracy: 0.6669 - val_loss: 0.9794 - learning_rate: 1.0000e-04
Epoch 7/60
1878/1878 ━━━━━━━━━

## 5. Resultados do experimento

In [5]:
display(results["fold_metrics"])
display(results["summary_metrics"])
print("Métricas OOF:", results["oof_metrics"])
print("Arquivos salvos em:", results["output_dir"])


,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,roc_auc_ovr_macro,fold,epochs_trained,training_samples_before_oversampling,training_samples_after_oversampling,validation_samples
0,0.706176,0.633797,0.550104,0.633797,0.575389,0.916691,1,19,6409,30037,1603
1,0.685590,0.628418,0.538059,0.628418,0.560257,0.932746,2,22,6409,30037,1603
2,0.725968,0.614211,0.565987,0.614211,0.561960,0.929134,3,15,6410,30037,1602
3,0.729089,0.662401,0.594531,0.662401,0.611429,0.943496,4,24,6410,30037,1602
4,0.735955,0.631049,0.563833,0.631049,0.588281,0.925731,5,24,6410,30044,1602


,mean,std
metric,,
accuracy,0.716555,0.020547
balanced_accuracy,0.633975,0.017593
precision_macro,0.562503,0.021160
recall_macro,0.633975,0.017593
f1_macro,0.579463,0.021163
roc_auc_ovr_macro,0.929560,0.009808


Métricas OOF: {'accuracy': 0.7165501747378932, 'balanced_accuracy': 0.6337504335518857, 'precision_macro': 0.5587521052443157, 'recall_macro': 0.6337504335518857, 'f1_macro': 0.5797500082556871, 'roc_auc_ovr_macro': 0.9293582214431261}
Arquivos salvos em: /Users/alexandre/Documents/GitHub/skin-cancer-classification/results/cross_validation/densenet169
